# 03 · Modelling: model families × imbalance strategies
Run with `python -m src.run_experiments`, `python -m src.train_final` and `python -m src.run_ablations`.

In [1]:
import sys, json; sys.path.insert(0, '..')
import pandas as pd
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40, 'display.width', 160)
T = '../reports/tables/'; F = '../reports/figures/'
g = pd.read_csv(T+'experiment_grid.csv')
g[g.decision=='tuned'][['model','imbalance','val_macro_f1','test_macro_f1','test_recall_Fatal','test_recall_Serious','test_roc_auc_ovr','test_accuracy','fit_seconds']].sort_values('test_macro_f1', ascending=False).round(3)

,model,imbalance,val_macro_f1,test_macro_f1,test_recall_Fatal,test_recall_Serious,test_roc_auc_ovr,test_accuracy,fit_seconds
27,xgb,none,0.410,0.407,0.150,0.280,0.689,0.733,21.9
31,xgb,undersample,0.410,0.407,0.155,0.271,0.685,0.736,294.7
25,lgbm,smote,0.409,0.405,0.165,0.271,0.682,0.733,205.7
19,lgbm,none,0.406,0.405,0.146,0.286,0.685,0.730,23.6
23,lgbm,undersample,0.408,0.405,0.096,0.294,0.681,0.734,102.6
13,rf,none,0.402,0.402,0.201,0.334,0.686,0.698,108.5
17,rf,undersample,0.402,0.402,0.199,0.274,0.684,0.722,37.4
33,xgb,smote,0.407,0.402,0.143,0.219,0.688,0.755,247.4
3,logreg,none,0.401,0.398,0.123,0.235,0.667,0.746,19.2
15,rf,class_weight,0.385,0.392,0.100,0.501,0.654,0.638,73.2


Plain argmax versus tuned decision scales (fitted on 2015 only):

In [2]:
g[g.decision=='argmax'][['model','imbalance','val_macro_f1','test_macro_f1','test_recall_Fatal','test_accuracy']].round(3)

,model,imbalance,val_macro_f1,test_macro_f1,test_recall_Fatal,test_accuracy
0,dummy,none,0.305,0.301,0.000,0.821
2,logreg,none,0.306,0.301,0.000,0.821
4,logreg,class_weight,0.334,0.337,0.479,0.517
6,logreg,smote,0.336,0.345,0.444,0.544
8,tree,none,0.311,0.306,0.000,0.820
10,tree,class_weight,0.331,0.333,0.445,0.523
12,rf,none,0.305,0.301,0.000,0.821
14,rf,class_weight,0.363,0.367,0.489,0.605
16,rf,undersample,0.322,0.316,0.000,0.819
18,lgbm,none,0.307,0.303,0.001,0.821


In [3]:
r = json.load(open(T+'final_results.json')); {k: r[k]['test_tuned'] for k in ('lgbm','xgb')}

{'lgbm': {'accuracy': 0.7320729090063184,
  'balanced_accuracy': 0.4144806116190181,
  'macro_f1': 0.4121123755398126,
  'weighted_f1': 0.7397275459087334,
  'mcc': 0.1533942116626151,
  'precision_Slight': 0.8544748923280799,
  'recall_Slight': 0.8247466949284168,
  'f1_Slight': 0.8393476461891789,
  'precision_Serious': 0.27303041336885076,
  'recall_Serious': 0.32198793138517867,
  'f1_Serious': 0.2954950844153151,
  'precision_Fatal': 0.10678021618080577,
  'recall_Fatal': 0.09670720854345892,
  'f1_Fatal': 0.10149439601494396,
  'recall_KSI': 0.3542777941114705,
  'roc_auc_ovr': 0.6982858822407514,
  'pr_auc_Serious': 0.25299513310520544,
  'pr_auc_Fatal': 0.0581740806328315},
 'xgb': {'accuracy': 0.722951779201873,
  'balanced_accuracy': 0.4341232972721251,
  'macro_f1': 0.4121900959129758,
  'weighted_f1': 0.7353166436468418,
  'mcc': 0.15276821051646453,
  'precision_Slight': 0.8565063802177204,
  'recall_Slight': 0.8140891524215873,
  'f1_Slight': 0.8347592693648983,
  'precis

## Cross-validation and significance

In [4]:
pd.read_csv(T+'cv_results.csv').round(4)

,model,cv_macro_f1_mean,cv_macro_f1_std,fold1,fold2,fold3,fold4,fold5,paired_t_vs_best,p_value_vs_best,best
0,Logistic Regression,0.3242,0.0028,0.3240,0.3284,0.3258,0.3225,0.3203,36.2207,0.0000,False
1,Decision Tree,0.3171,0.0065,0.3152,0.3149,0.3226,0.3258,0.3072,27.0766,0.0000,False
2,Random Forest,0.3535,0.0029,0.3574,0.3546,0.3549,0.3517,0.3491,22.3556,0.0000,False
3,XGBoost,0.3725,0.0029,0.3719,0.3736,0.3755,0.3744,0.3671,9.7481,0.0006,False
4,LightGBM,0.3813,0.0023,0.3824,0.3794,0.3842,0.3827,0.3780,NaN,NaN,True


## Ablations

In [5]:
pd.read_csv(T+'ablations.csv').round(4)

,variant,feature_set,macro_f1,accuracy,balanced_accuracy,recall_Fatal,recall_Serious,roc_auc_ovr,pr_auc_Fatal,mcc
0,Full model (scene + time + location + spatial ...,full,0.4117,0.7445,0.4199,0.1371,0.2735,0.6940,0.0569,0.1499
1,- spatial history features,no_history,0.4070,0.7514,0.4056,0.0937,0.2628,0.6880,0.0544,0.1445
2,- time features,no_time,0.4089,0.7416,0.4177,0.1338,0.2737,0.6875,0.0512,0.1456
3,- raw coordinates (lat/lon),no_geo,0.4104,0.7416,0.4200,0.1379,0.2772,0.6935,0.0561,0.1492
4,"Scene only (no history, no coordinates)",scene_only,0.4059,0.7507,0.4049,0.0943,0.2606,0.6876,0.0542,0.1426
5,+ road-level history,with_road_history,0.4130,0.7441,0.4212,0.1400,0.2753,0.6943,0.0566,0.1503
6,+ LEAKAGE: casualties & police attendance,with_leaky,0.4215,0.7488,0.4301,0.1486,0.2912,0.7014,0.0634,0.1689


## Other protocols
Stratified random split and the binary Killed/Seriously-injured task.

In [6]:
{k: {m: round(v, 4) for m, v in r[k]['unweighted_argmax' if k=='random_split' else 'tuned'].items() if m in ('accuracy','macro_f1','roc_auc_ovr','roc_auc','recall_KSI')} for k in ('random_split','binary_ksi')}

{'random_split': {'accuracy': 0.8459,
  'macro_f1': 0.3058,
  'recall_KSI': 0.0004,
  'roc_auc_ovr': 0.7118},
 'binary_ksi': {'accuracy': 0.7534,
  'macro_f1': 0.5866,
  'recall_KSI': 0.331,
  'roc_auc': 0.6578}}